In [ ]:
import zipfile
import os
from google.colab import files

print("Please select your dataset ZIP file:")
uploaded = files.upload()

for filename in uploaded.keys():
    if filename.endswith(".zip"):
        print(f"Extracting {filename}...")
        with zipfile.ZipFile(filename, 'r') as zip_ref:
            zip_ref.extractall(".")
        print("Extraction complete!")

for root, dirs, filenames in os.walk("."):
    for file in filenames:
        if file.endswith(".csv") and root != ".":
            source_path = os.path.join(root, file)
            target_path = os.path.join(".", file)
            os.rename(source_path, target_path)

print("\nAvailable CSV files in current working directory:")
print([f for f in os.listdir(".") if f.endswith(".csv")])

Please select your dataset ZIP file:


Saving Assessment 1-20261030T033729Z-1-001.zip to Assessment 1-20261030T033729Z-1-001.zip
Extracting Assessment 1-20261030T033729Z-1-001.zip...
Extraction complete!

Available CSV files in current working directory:
['california_housing_test.csv', 'meter_readings_90000.csv', 'california_housing_train.csv', 'mnist_test.csv', 'maintenance_tickets_1600.csv', 'tariff_master.csv', 'meter_readings_extra_12000.csv', 'meter_master.csv', 'substation_master.csv', 'mnist_train_small.csv']


In [ ]:
from google.colab import files
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.window import Window
from pyspark import StorageLevel

spark = SparkSession.builder \
    .appName("Energy Analytics Assessment - Set 03") \
    .master("local[*]") \
    .getOrCreate()

def load_csv(filename):
    return spark.read.csv(filename, header=True, inferSchema=True)

SECTION 01 / DATA PREPARATION & ANALYSIS

In [ ]:
readings_df = load_csv("meter_readings_90000.csv")
readings_extra_df = load_csv("meter_readings_extra_12000.csv")
meter_master_df = load_csv("meter_master.csv")
substation_master_df = load_csv("substation_master.csv")
tariff_master_df = load_csv("tariff_master.csv")
tickets_df = load_csv("maintenance_tickets_1600.csv")

print("Schema of Main Meter Readings:")
readings_df.printSchema()

print(f"Total Row Count: {readings_df.count()}")
readings_df.show(10, truncate=False)

Schema of Main Meter Readings:
root
 |-- reading_id: string (nullable = true)
 |-- meter_id: string (nullable = true)
 |-- reading_timestamp: timestamp (nullable = true)
 |-- energy_kwh: double (nullable = true)
 |-- voltage_v: double (nullable = true)
 |-- power_factor: double (nullable = true)
 |-- outage_minutes: integer (nullable = true)
 |-- reading_status: string (nullable = true)
 |-- source_type: string (nullable = true)
 |-- ambient_temperature_c: double (nullable = true)
 |-- operator_id: string (nullable = true)

Total Row Count: 90000
+----------+--------+-------------------+----------+---------+------------+--------------+--------------+-----------+---------------------+-----------+
|reading_id|meter_id|reading_timestamp  |energy_kwh|voltage_v|power_factor|outage_minutes|reading_status|source_type|ambient_temperature_c|operator_id|
+----------+--------+-------------------+----------+---------+------------+--------------+--------------+-----------+---------------------+----

In [ ]:
selected_df = readings_df.select(
    "reading_id", "meter_id", "reading_timestamp", "energy_kwh", "reading_status"
)

renamed_df = readings_df.withColumnRenamed("source_type", "collection_mode")
renamed_df.show(5)

+----------+--------+-------------------+----------+---------+------------+--------------+--------------+---------------+---------------------+-----------+
|reading_id|meter_id|  reading_timestamp|energy_kwh|voltage_v|power_factor|outage_minutes|reading_status|collection_mode|ambient_temperature_c|operator_id|
+----------+--------+-------------------+----------+---------+------------+--------------+--------------+---------------+---------------------+-----------+
| RD0000001|   M0082|2026-04-01 02:00:00|     22.53|    213.1|       0.939|             0|         Valid|     SmartMeter|                 36.2|      OP029|
| RD0000002|   M0020|2026-01-31 06:00:00|      2.97|    227.4|       0.845|             0|         Valid|     SmartMeter|                 41.2|      OP025|
| RD0000003|   M0054|2026-04-11 10:15:00|      8.61|    239.3|       0.879|             0|     Estimated|     SmartMeter|                 20.9|      OP007|
| RD0000004|   M0111|2026-03-14 00:00:00|      1.87|    246.5|  

In [ ]:
valid_high_energy_df = readings_df.filter(
    (F.col("reading_status") == "Valid") & (F.col("energy_kwh") > 20)
).orderBy(F.col("energy_kwh").desc())

valid_high_energy_df.show(10)

+----------+--------+-------------------+----------+---------+------------+--------------+--------------+-----------+---------------------+-----------+
|reading_id|meter_id|  reading_timestamp|energy_kwh|voltage_v|power_factor|outage_minutes|reading_status|source_type|ambient_temperature_c|operator_id|
+----------+--------+-------------------+----------+---------+------------+--------------+--------------+-----------+---------------------+-----------+
| RD0041703|   M0101|2026-03-06 22:30:00|     110.0|    213.1|       0.822|             0|         Valid| SmartMeter|                 31.1|      OP010|
| RD0077494|   M0014|2026-05-05 17:30:00|    109.99|     NULL|       0.771|             0|         Valid|     Manual|                 39.0|      OP031|
| RD0039063|   M0101|2026-02-01 00:45:00|    109.96|    231.6|       0.835|             0|         Valid| SmartMeter|                 20.1|      OP021|
| RD0034412|   M0026|2026-02-24 21:15:00|    109.96|    231.6|       0.964|             

In [ ]:
energy_band_df = readings_df.withColumn(
    "energy_band",
    F.when(F.col("energy_kwh").isNull(), "Unknown")
     .when(F.col("energy_kwh") < 5, "Low")
     .when((F.col("energy_kwh") >= 5) & (F.col("energy_kwh") <= 25), "Medium")
     .when(F.col("energy_kwh") > 25, "High")
     .otherwise("Unknown")
)

energy_band_df.show(10)

+----------+--------+-------------------+----------+---------+------------+--------------+--------------+-----------+---------------------+-----------+-----------+
|reading_id|meter_id|  reading_timestamp|energy_kwh|voltage_v|power_factor|outage_minutes|reading_status|source_type|ambient_temperature_c|operator_id|energy_band|
+----------+--------+-------------------+----------+---------+------------+--------------+--------------+-----------+---------------------+-----------+-----------+
| RD0000001|   M0082|2026-04-01 02:00:00|     22.53|    213.1|       0.939|             0|         Valid| SmartMeter|                 36.2|      OP029|     Medium|
| RD0000002|   M0020|2026-01-31 06:00:00|      2.97|    227.4|       0.845|             0|         Valid| SmartMeter|                 41.2|      OP025|        Low|
| RD0000003|   M0054|2026-04-11 10:15:00|      8.61|    239.3|       0.879|             0|     Estimated| SmartMeter|                 20.9|      OP007|     Medium|
| RD0000004|   M

In [ ]:
null_counts = readings_df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in ["energy_kwh", "voltage_v", "power_factor", "operator_id"]
])

null_counts.show()

+----------+---------+------------+-----------+
|energy_kwh|voltage_v|power_factor|operator_id|
+----------+---------+------------+-----------+
|      1620|     2070|        1440|       5400|
+----------+---------+------------+-----------+



In [ ]:
filled_readings_df = readings_df.na.fill({
    "operator_id": "UNASSIGNED",
    "power_factor": 0.90
})

filled_readings_df.show(10)

+----------+--------+-------------------+----------+---------+------------+--------------+--------------+-----------+---------------------+-----------+
|reading_id|meter_id|  reading_timestamp|energy_kwh|voltage_v|power_factor|outage_minutes|reading_status|source_type|ambient_temperature_c|operator_id|
+----------+--------+-------------------+----------+---------+------------+--------------+--------------+-----------+---------------------+-----------+
| RD0000001|   M0082|2026-04-01 02:00:00|     22.53|    213.1|       0.939|             0|         Valid| SmartMeter|                 36.2|      OP029|
| RD0000002|   M0020|2026-01-31 06:00:00|      2.97|    227.4|       0.845|             0|         Valid| SmartMeter|                 41.2|      OP025|
| RD0000003|   M0054|2026-04-11 10:15:00|      8.61|    239.3|       0.879|             0|     Estimated| SmartMeter|                 20.9|      OP007|
| RD0000004|   M0111|2026-03-14 00:00:00|      1.87|    246.5|       0.831|             

In [ ]:
initial_count = readings_df.count()
cleaned_readings_df = readings_df.filter(
    F.col("energy_kwh").isNotNull() & F.col("voltage_v").isNotNull()
)
final_count = cleaned_readings_df.count()

print(f"Initial Rows: {initial_count}")
print(f"Final Rows: {final_count}")
print(f"Rows Removed: {initial_count - final_count}")

Initial Rows: 90000
Final Rows: 86349
Rows Removed: 3651


In [ ]:
valid_energy_by_site = readings_df.filter(F.col("reading_status") == "Valid") \
    .join(meter_master_df, "meter_id", "inner") \
    .groupBy("site_type") \
    .agg(F.sum("energy_kwh").alias("total_valid_energy_kwh")) \
    .orderBy("site_type")

valid_energy_by_site.show()

+---------------+----------------------+
|      site_type|total_valid_energy_kwh|
+---------------+----------------------+
|     Commercial|     451525.1799999996|
|     Industrial|     710317.4100000021|
|Public Facility|             138870.41|
|    Residential|     100863.2900000004|
+---------------+----------------------+



In [ ]:
valid_energy_region_city = readings_df.filter(F.col("reading_status") == "Valid") \
    .join(meter_master_df, "meter_id", "inner") \
    .join(substation_master_df, "substation_id", "inner") \
    .groupBy("region", "city") \
    .agg(F.sum("energy_kwh").alias("total_valid_energy_kwh")) \
    .orderBy("region", "city")

valid_energy_region_city.show()

+-------+-------------+----------------------+
| region|         city|total_valid_energy_kwh|
+-------+-------------+----------------------+
|Central|       Indore|    176503.91000000012|
|  North|        Delhi|     94234.89000000014|
|  North|       Jaipur|     47863.15999999996|
|  North|      Lucknow|     142610.3999999998|
|  South|    Bengaluru|     73787.59999999998|
|  South|      Chennai|     51829.05999999996|
|  South|    Hyderabad|    51937.789999999884|
|  South|        Kochi|     204272.4600000002|
|  South|Visakhapatnam|    205015.82999999932|
|   West|    Ahmedabad|     138627.7400000004|
|   West|       Mumbai|     52116.20999999999|
|   West|         Pune|    162777.23999999996|
+-------+-------------+----------------------+



In [ ]:
energy_charges_df = readings_df.filter(
    (F.col("reading_status") != "Rejected") & (F.col("energy_kwh").isNotNull())
) \
    .join(meter_master_df, "meter_id", "inner") \
    .join(tariff_master_df, "tariff_id", "inner") \
    .withColumn("energy_charge_inr", F.col("energy_kwh") * F.col("rate_per_kwh_inr"))

energy_charges_df.select(
    "reading_id", "meter_id", "energy_kwh", "tariff_id", "rate_per_kwh_inr", "energy_charge_inr"
).show(10)

+----------+--------+----------+-------------------+----------------+------------------+
|reading_id|meter_id|energy_kwh|          tariff_id|rate_per_kwh_inr| energy_charge_inr|
+----------+--------+----------+-------------------+----------------+------------------+
| RD0000001|   M0082|     22.53|2026-10-30 05:00:00|             6.7|150.95100000000002|
| RD0000002|   M0020|      2.97|2026-10-30 02:00:00|             6.4|19.008000000000003|
| RD0000003|   M0054|      8.61|2026-10-30 03:00:00|             8.2| 70.60199999999999|
| RD0000004|   M0111|      1.87|2026-10-30 01:00:00|             5.1| 9.536999999999999|
| RD0000005|   M0135|      4.39|2026-10-30 03:00:00|             8.2|            35.998|
| RD0000006|   M0139|      0.36|2026-10-30 02:00:00|             6.4|             2.304|
| RD0000007|   M0142|     79.47|2026-10-30 04:00:00|             9.1| 723.1769999999999|
| RD0000009|   M0005|      1.96|2026-10-30 01:00:00|             5.1| 9.995999999999999|
| RD0000010|   M0052|

In [ ]:
tickets_summary = tickets_df.join(meter_master_df, "meter_id", "inner") \
    .groupBy("site_type", "severity") \
    .agg(
        F.count("ticket_id").alias("ticket_count"),
        F.avg("maintenance_cost_inr").alias("avg_maintenance_cost_inr")
    ) \
    .orderBy("site_type", "severity")

tickets_summary.show()

+---------------+--------+------------+------------------------+
|      site_type|severity|ticket_count|avg_maintenance_cost_inr|
+---------------+--------+------------+------------------------+
|     Commercial|Critical|          37|      13424.726216216219|
|     Commercial|    High|         130|      12293.073793103451|
|     Commercial|     Low|         166|      12223.517692307689|
|     Commercial|  Medium|         171|      13706.600886075952|
|     Industrial|Critical|          17|      12296.198000000002|
|     Industrial|    High|          63|      12838.869344262299|
|     Industrial|     Low|          66|       12717.52508474576|
|     Industrial|  Medium|         106|      12043.898526315788|
|Public Facility|Critical|          15|       8986.843571428572|
|Public Facility|    High|          48|      12905.279347826086|
|Public Facility|     Low|          60|      11195.929464285715|
|Public Facility|  Medium|          97|      13050.628421052634|
|    Residential|Critical

In [ ]:
substation_reliability = readings_df \
    .join(meter_master_df, "meter_id", "inner") \
    .groupBy("substation_id") \
    .agg(\
        F.avg("voltage_v").alias("avg_voltage_v"),\
        F.sum("outage_minutes").alias("total_outage_minutes")\
    ) \
    .orderBy(F.col("total_outage_minutes").desc())

substation_reliability.show(5)

+-------------+------------------+--------------------+
|substation_id|     avg_voltage_v|total_outage_minutes|
+-------------+------------------+--------------------+
|        SUB10|230.04010567581344|               91224|
|        SUB09|230.04586272640606|               81734|
|        SUB04| 230.0558687466385|               75872|
|        SUB08|229.88012557077673|               75581|
|        SUB03|230.03094896227708|               61662|
+-------------+------------------+--------------------+
only showing top 5 rows


In [ ]:
status_by_source = readings_df \
    .groupBy("source_type") \
    .pivot("reading_status", ["Valid", "Estimated", "Rejected"]) \
    .count() \
    .withColumn("total_count", F.coalesce(F.col("Valid"), F.lit(0)) +
                               F.coalesce(F.col("Estimated"), F.lit(0)) +
                               F.coalesce(F.col("Rejected"), F.lit(0))) \
    .orderBy(F.col("total_count").desc())

status_by_source.show()

+-----------+-----+---------+--------+-----------+
|source_type|Valid|Estimated|Rejected|total_count|
+-----------+-----+---------+--------+-----------+
| SmartMeter|65521|    10443|    3202|      79166|
|     Manual| 8965|     1405|     464|      10834|
+-----------+-----+---------+--------+-----------+



SECTION 02 / ADVANCED DATAFRAME OPERATIONS

In [ ]:
readings_with_date = readings_df \
    .withColumn("reading_timestamp", F.to_timestamp("reading_timestamp")) \
    .withColumn("year", F.year("reading_timestamp")) \
    .withColumn("month", F.month("reading_timestamp"))

monthly_regional_energy = readings_with_date \
    .filter(F.col("reading_status") == "Valid") \
    .join(meter_master_df, "meter_id", "inner") \
    .join(substation_master_df, "substation_id", "inner") \
    .groupBy("year", "month", "region") \
    .agg(F.sum("energy_kwh").alias("monthly_valid_energy_kwh")) \
    .orderBy("year", "month", "region")

monthly_regional_energy.show(10)

+----+-----+-------+------------------------+
|year|month| region|monthly_valid_energy_kwh|
+----+-----+-------+------------------------+
|2026|    1|Central|       31475.57999999999|
|2026|    1|  North|      48849.309999999925|
|2026|    1|  South|       97547.38999999987|
|2026|    1|   West|       59967.56999999997|
|2026|    2|Central|      28161.410000000007|
|2026|    2|  North|      45444.249999999985|
|2026|    2|  South|       90864.80000000006|
|2026|    2|   West|       55807.89999999989|
|2026|    3|Central|                29805.85|
|2026|    3|  North|       51094.61000000006|
+----+-----+-------+------------------------+
only showing top 10 rows


In [ ]:
valid_meter_energy = readings_df \
    .filter(F.col("reading_status") == "Valid") \
    .join(meter_master_df, "meter_id", "inner") \
    .groupBy("substation_id", "meter_id") \
    .agg(F.sum("energy_kwh").alias("total_meter_energy"))

window_substation = Window.partitionBy("substation_id").orderBy(F.col("total_meter_energy").desc())

top_meters_per_substation = valid_meter_energy \
    .withColumn("rank", F.dense_rank().over(window_substation)) \
    .filter(F.col("rank") <= 3) \
    .orderBy("substation_id", "rank")

top_meters_per_substation.show(15)

+-------------+--------+------------------+----+
|substation_id|meter_id|total_meter_energy|rank|
+-------------+--------+------------------+----+
|        SUB01|   M0130|           9364.98|   1|
|        SUB01|   M0110| 9259.750000000002|   2|
|        SUB01|   M0068| 8832.580000000005|   3|
|        SUB02|   M0071|          29242.04|   1|
|        SUB02|   M0145|28641.319999999992|   2|
|        SUB02|   M0133| 7060.140000000003|   3|
|        SUB03|   M0096| 9632.740000000003|   1|
|        SUB03|   M0080| 9525.940000000002|   2|
|        SUB03|   M0073| 9369.789999999995|   3|
|        SUB04|   M0038|          29505.88|   1|
|        SUB04|   M0099|29216.319999999996|   2|
|        SUB04|   M0098|28613.460000000014|   3|
|        SUB05|   M0094| 9659.639999999996|   1|
|        SUB05|   M0050| 9570.859999999997|   2|
|        SUB05|   M0008| 9524.780000000004|   3|
+-------------+--------+------------------+----+
only showing top 15 rows


In [ ]:
window_meter_time = Window.partitionBy("meter_id") \
    .orderBy(F.col("reading_timestamp").asc(), F.col("reading_id").asc()) \
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)

cumulative_energy_df = readings_with_date \
    .filter(F.col("reading_status") == "Valid") \
    .withColumn("cumulative_valid_energy", F.sum("energy_kwh").over(window_meter_time))

cumulative_energy_df.select("meter_id", "reading_timestamp", "reading_id", "energy_kwh", "cumulative_valid_energy").show(10)

+--------+-------------------+----------+----------+-----------------------+
|meter_id|  reading_timestamp|reading_id|energy_kwh|cumulative_valid_energy|
+--------+-------------------+----------+----------+-----------------------+
|   M0001|2026-01-01 01:15:00| RD0083587|     22.27|                  22.27|
|   M0001|2026-01-01 03:30:00| RD0086275|      4.05|                  26.32|
|   M0001|2026-01-01 09:45:00| RD0088204|     11.28|                   37.6|
|   M0001|2026-01-02 04:45:00| RD0005149|     11.76|                  49.36|
|   M0001|2026-01-02 08:15:00| RD0035530|     10.06|                  59.42|
|   M0001|2026-01-02 18:30:00| RD0070104|     15.83|                  75.25|
|   M0001|2026-01-03 00:30:00| RD0006677|     17.06|                  92.31|
|   M0001|2026-01-03 06:00:00| RD0021042|     17.14|                 109.45|
|   M0001|2026-01-03 08:30:00| RD0032205|     20.46|                 129.91|
|   M0001|2026-01-03 18:15:00| RD0074606|      20.4|                 150.31|

In [ ]:
count_main = readings_df.count()
count_extra = readings_extra_df.count()

combined_readings_df = readings_df.unionByName(readings_extra_df)
count_combined = combined_readings_df.count()

print(f"Main Readings Count: {count_main}")
print(f"Extra Readings Count: {count_extra}")
print(f"Expected Sum Count: {count_main + count_extra}")
print(f"Combined Union Count: {count_combined}")

Main Readings Count: 90000
Extra Readings Count: 12000
Expected Sum Count: 102000
Combined Union Count: 102000


In [ ]:
duplicate_readings = combined_readings_df.groupBy("reading_id") \
    .count() \
    .filter(F.col("count") > 1)

print(f"Number of duplicate reading_id values: {duplicate_readings.count()}")

deduped_readings_df = combined_readings_df.dropDuplicates(["reading_id"])
print(f"Final Unique Reading Count: {deduped_readings_df.count()}")

Number of duplicate reading_id values: 700
Final Unique Reading Count: 101300


In [ ]:
lazy_pipeline = readings_df \
    .filter(F.col("reading_status") == "Valid") \
    .withColumn("energy_band", F.when(F.col("energy_kwh") > 20, "High").otherwise("Standard")) \
    .groupBy("meter_id", "energy_band") \
    .agg(F.avg("energy_kwh").alias("avg_energy")) \
    .orderBy(F.col("avg_energy").desc())

lazy_pipeline.explain(True)

== Parsed Logical Plan ==
'Sort ['avg_energy DESC NULLS LAST], true
+- Aggregate [meter_id#18, energy_band#1060], [meter_id#18, energy_band#1060, avg(energy_kwh#20) AS avg_energy#1061]
   +- Project [reading_id#17, meter_id#18, reading_timestamp#19, energy_kwh#20, voltage_v#21, power_factor#22, outage_minutes#23, reading_status#24, source_type#25, ambient_temperature_c#26, operator_id#27, CASE WHEN (energy_kwh#20 > cast(20 as double)) THEN High ELSE Standard END AS energy_band#1060]
      +- Filter (reading_status#24 = Valid)
         +- Relation [reading_id#17,meter_id#18,reading_timestamp#19,energy_kwh#20,voltage_v#21,power_factor#22,outage_minutes#23,reading_status#24,source_type#25,ambient_temperature_c#26,operator_id#27] csv

== Analyzed Logical Plan ==
meter_id: string, energy_band: string, avg_energy: double
Sort [avg_energy#1061 DESC NULLS LAST], true
+- Aggregate [meter_id#18, energy_band#1060], [meter_id#18, energy_band#1060, avg(energy_kwh#20) AS avg_energy#1061]
   +- Proje

In [ ]:
import time

valid_readings_df = readings_df.filter(F.col("reading_status") == "Valid").cache()

# First action
start_time = time.time()
count1 = valid_readings_df.count()
time_1 = time.time() - start_time

# Second action
start_time = time.time()
count2 = valid_readings_df.count()
time_2 = time.time() - start_time

print(f"First count execution time: {time_1:.4f} seconds")
print(f"Second count execution time: {time_2:.4f} seconds")
print(f"Is cached: {valid_readings_df.is_cached}")

valid_readings_df.unpersist()

First count execution time: 2.2979 seconds
Second count execution time: 0.2581 seconds
Is cached: True


DataFrame[reading_id: string, meter_id: string, reading_timestamp: timestamp, energy_kwh: double, voltage_v: double, power_factor: double, outage_minutes: int, reading_status: string, source_type: string, ambient_temperature_c: double, operator_id: string]

In [ ]:
outage_df = readings_df.filter(F.col("outage_minutes") > 0) \
    .persist(StorageLevel.MEMORY_AND_DISK)

# Action 1
total_outages = outage_df.count()
# Action 2
avg_outage_time = outage_df.select(F.avg("outage_minutes")).collect()[0][0]

print(f"Total Outage Events: {total_outages}")
print(f"Average Outage Minutes: {avg_outage_time:.2f}")

outage_df.unpersist()

Total Outage Events: 7653
Average Outage Minutes: 92.47


DataFrame[reading_id: string, meter_id: string, reading_timestamp: timestamp, energy_kwh: double, voltage_v: double, power_factor: double, outage_minutes: int, reading_status: string, source_type: string, ambient_temperature_c: double, operator_id: string]

In [ ]:
# Uncached test
uncached_agg = readings_df.groupBy("meter_id").agg(F.sum("energy_kwh"))
t0 = time.time(); uncached_agg.count(); t_uncached_1 = time.time() - t0
t0 = time.time(); uncached_agg.count(); t_uncached_2 = time.time() - t0

# Cached test
cached_agg = readings_df.groupBy("meter_id").agg(F.sum("energy_kwh")).cache()
t0 = time.time(); cached_agg.count(); t_cached_1 = time.time() - t0
t0 = time.time(); cached_agg.count(); t_cached_2 = time.time() - t0

print(f"Uncached Run 1: {t_uncached_1:.4f}s | Uncached Run 2: {t_uncached_2:.4f}s")
print(f"Cached Run 1: {t_cached_1:.4f}s   | Cached Run 2: {t_cached_2:.4f}s")

cached_agg.unpersist()

Uncached Run 1: 0.8130s | Uncached Run 2: 0.4596s
Cached Run 1: 5.2624s   | Cached Run 2: 2.9639s


DataFrame[meter_id: string, sum(energy_kwh): double]

In [ ]:
repartitioned_8 = readings_df.repartition(8)

rows_per_partition = repartitioned_8.withColumn("partition_id", F.spark_partition_id()) \
    .groupBy("partition_id") \
    .count() \
    .orderBy("partition_id")

rows_per_partition.show()

+------------+-----+
|partition_id|count|
+------------+-----+
|           0|11249|
|           1|11249|
|           2|11251|
|           3|11251|
|           4|11250|
|           5|11250|
|           6|11250|
|           7|11250|
+------------+-----+



In [ ]:
df_12_parts = readings_df.repartition(12)

coalesced_3 = df_12_parts.coalesce(3)
repartitioned_3 = df_12_parts.repartition(3)

coalesced_3.explain()

repartitioned_3.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Coalesce 3
   +- Exchange RoundRobinPartitioning(12), REPARTITION_BY_NUM, [plan_id=2868]
      +- FileScan csv [reading_id#17,meter_id#18,reading_timestamp#19,energy_kwh#20,voltage_v#21,power_factor#22,outage_minutes#23,reading_status#24,source_type#25,ambient_temperature_c#26,operator_id#27] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/content/meter_readings_90000.csv], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<reading_id:string,meter_id:string,reading_timestamp:timestamp,energy_kwh:double,voltage_v:...


== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Exchange RoundRobinPartitioning(3), REPARTITION_BY_NUM, [plan_id=2877]
   +- FileScan csv [reading_id#17,meter_id#18,reading_timestamp#19,energy_kwh#20,voltage_v#21,power_factor#22,outage_minutes#23,reading_status#24,source_type#25,ambient_temperature_c#26,operator_id#27] Batched: false, DataFilters: [], Fo

In [ ]:
enriched_df = cleaned_readings_df \
    .join(meter_master_df, "meter_id", "left") \
    .join(substation_master_df, "substation_id", "left") \
    .repartition(4)

output_parquet_path = "enriched_meter_readings.parquet"
enriched_df.write.mode("overwrite").parquet(output_parquet_path)

import os
print("Generated Parquet Files:")
print(os.listdir(output_parquet_path))

Generated Parquet Files:
['_SUCCESS', '.part-00000-b4041662-2767-41ee-8d5a-77a827fe1672-c000.snappy.parquet.crc', '.part-00003-b4041662-2767-41ee-8d5a-77a827fe1672-c000.snappy.parquet.crc', 'part-00001-b4041662-2767-41ee-8d5a-77a827fe1672-c000.snappy.parquet', '.part-00001-b4041662-2767-41ee-8d5a-77a827fe1672-c000.snappy.parquet.crc', '.part-00002-b4041662-2767-41ee-8d5a-77a827fe1672-c000.snappy.parquet.crc', '._SUCCESS.crc', 'part-00000-b4041662-2767-41ee-8d5a-77a827fe1672-c000.snappy.parquet', 'part-00003-b4041662-2767-41ee-8d5a-77a827fe1672-c000.snappy.parquet', 'part-00002-b4041662-2767-41ee-8d5a-77a827fe1672-c000.snappy.parquet']


SECTION 03 / INTEGRATED ASSESSMENT: FINAL CAPSTONE


In [ ]:
combined_df = readings_df.unionByName(readings_extra_df)
deduped_df = combined_df.dropDuplicates(["reading_id"])

In [ ]:
cleaned_capstone_df = deduped_df \
    .filter(F.col("energy_kwh").isNotNull()) \
    .withColumn("is_billable", F.when(F.col("reading_status") != "Rejected", True).otherwise(False))

In [ ]:
enriched_capstone_df = cleaned_capstone_df \
    .join(meter_master_df, "meter_id", "inner") \
    .join(substation_master_df, "substation_id", "inner") \
    .join(tariff_master_df, "tariff_id", "inner")

In [ ]:
monthly_report_df = enriched_capstone_df \
    .withColumn("month", F.date_format(F.to_timestamp("reading_timestamp"), "yyyy-MM")) \
    .withColumn("billable_kwh", F.when(F.col("is_billable"), F.col("energy_kwh")).otherwise(0.0)) \
    .withColumn("billable_charge_inr", F.col("billable_kwh") * F.col("rate_per_kwh_inr")) \
    .withColumn("has_outage", F.when(F.col("outage_minutes") > 0, 1).otherwise(0)) \
    .groupBy("region", "site_type", "month") \
    .agg(\
        F.sum("energy_kwh").alias("total_kwh"),\
        F.sum("billable_kwh").alias("total_billable_kwh"),\
        F.sum("billable_charge_inr").alias("total_billable_charge_inr"),\
        F.sum("outage_minutes").alias("total_outage_minutes"),\
        (F.sum("has_outage") / F.count("reading_id") * 100).alias("outage_reading_percentage")\
    )


In [ ]:
region_window = Window.partitionBy("region", "month").orderBy(F.col("total_billable_charge_inr").desc())

final_report_df = monthly_report_df \
    .withColumn("site_type_rank", F.dense_rank().over(region_window)) \
    .orderBy("region", "month", "site_type_rank")

In [ ]:
final_report_df.persist(StorageLevel.MEMORY_AND_DISK)

print("--- Capstone Report Action 1: Total Row Count ---")
print(f"Report Rows: {final_report_df.count()}")

print("\n--- Capstone Report Action 2: Top Regional Sites Sample ---")
final_report_df.show(10, truncate=False)

final_report_df.unpersist()

--- Capstone Report Action 1: Total Row Count ---
Report Rows: 120

--- Capstone Report Action 2: Top Regional Sites Sample ---
+-------+-----------+-------+------------------+------------------+-------------------------+--------------------+-------------------------+--------------+
|region |site_type  |month  |total_kwh         |total_billable_kwh|total_billable_charge_inr|total_outage_minutes|outage_reading_percentage|site_type_rank|
+-------+-----------+-------+------------------+------------------+-------------------------+--------------------+-------------------------+--------------+
|Central|Industrial |2026-01|25618.800000000003|24871.22          |226328.102               |4396                |11.342592592592593       |1             |
|Central|Commercial |2026-01|11063.699999999993|10649.029999999995|87322.04599999999        |5305                |10.238907849829351       |2             |
|Central|Residential|2026-01|843.2             |814.03            |4636.908000000001        

DataFrame[region: string, site_type: string, month: string, total_kwh: double, total_billable_kwh: double, total_billable_charge_inr: double, total_outage_minutes: bigint, outage_reading_percentage: double, site_type_rank: int]

In [ ]:
capstone_output_path = "monthly_energy_reliability_report.parquet"

final_report_df.repartition(2) \
    .write.mode("overwrite") \
    .parquet(capstone_output_path)

print(f"\nFinal Capstone report written to '{capstone_output_path}'")
print("Output Parquet Files:", os.listdir(capstone_output_path))


Final Capstone report written to 'monthly_energy_reliability_report.parquet'
Output Parquet Files: ['_SUCCESS', '.part-00001-35bfcfe0-7403-49df-ab22-3a88ff5eb99d-c000.snappy.parquet.crc', 'part-00001-35bfcfe0-7403-49df-ab22-3a88ff5eb99d-c000.snappy.parquet', '.part-00000-35bfcfe0-7403-49df-ab22-3a88ff5eb99d-c000.snappy.parquet.crc', '._SUCCESS.crc', 'part-00000-35bfcfe0-7403-49df-ab22-3a88ff5eb99d-c000.snappy.parquet']


In [ ]:
# Load the saved capstone Parquet report
view_report_df = spark.read.parquet(capstone_output_path)

# Display the top 20 rows of the report
view_report_df.show(20, truncate=False)

# Alternative: Convert to Pandas for a cleaner table layout inside Colab
display(view_report_df.limit(20).toPandas())

+-------+---------------+-------+------------------+------------------+-------------------------+--------------------+-------------------------+--------------+
|region |site_type      |month  |total_kwh         |total_billable_kwh|total_billable_charge_inr|total_outage_minutes|outage_reading_percentage|site_type_rank|
+-------+---------------+-------+------------------+------------------+-------------------------+--------------------+-------------------------+--------------+
|South  |Commercial     |2026-01|38297.88          |36720.51999999999 |301108.26399999997       |16942               |8.928571428571429        |2             |
|West   |Residential    |2026-04|6892.23           |6637.9400000000005|39536.95099999996        |14678               |8.041237113402062        |4             |
|North  |Industrial     |2026-01|31853.480000000003|30649.829999999994|278913.4530000001        |3936                |7.421150278293136        |1             |
|West   |Residential    |2026-08|2615.55

,region,site_type,month,total_kwh,total_billable_kwh,total_billable_charge_inr,total_outage_minutes,outage_reading_percentage,site_type_rank
0,South,Commercial,2026-01,38297.88,36720.52,301108.264,16942,8.928571,2
1,West,Residential,2026-04,6892.23,6637.94,39536.951,14678,8.041237,4
2,North,Industrial,2026-01,31853.48,30649.83,278913.453,3936,7.421150,1
3,West,Residential,2026-08,2615.55,2520.00,14947.223,6589,9.041096,4
4,Central,Residential,2026-07,271.81,252.71,1445.705,788,9.090909,3
5,North,Residential,2026-04,2374.38,2311.41,14384.733,4140,6.800618,4
6,Central,Residential,2026-01,843.20,814.03,4636.908,1869,8.658009,3
7,North,Residential,2026-07,866.76,819.80,5050.407,864,4.329004,4
8,South,Commercial,2026-04,39005.31,37162.90,304735.780,13887,7.863000,2
9,West,Residential,2026-07,2531.83,2438.25,14435.138,5116,7.725322,4
